**Executing this notebook to check zip file is good and extracting to right folder**

In [1]:
%run ../utility/common_functions.ipynb

In [ ]:

# Read inbound folder from configuration
# Convert configured stage reference into Snowflake stage path
input_stage_path = input_folder.replace("@AZURE_SNOW_STAGE",f"@{stage_name}")

# Discover ZIP files dynamically
zip_files = session.sql(f"LIST {input_stage_path}").collect()

# Extract only the file names, not the complete Azure URLs
zip_files = [
    row["name"].rsplit("/", 1)[-1]
    for row in zip_files
    if row["name"].lower().endswith(".zip")
]
print("ZIP files discovered:", len(zip_files))
for file_name in zip_files:
    print("ZIP filename:", file_name)

In [2]:
folder_name = datetime.now().strftime("%Y%m%d")
run_start_time = datetime.now()
run_id = (f"{datetime.now().strftime('%Y%m%d_%H%M%S')}")
pipeline_name = "Customer Billing Pipeline"
task_name = "Stage 1 Unzip Source File"

In [3]:
final_input_folder = (input_stage_path.rstrip("/") + "/" + file_name)
print("Final Snowflake ZIP path:", final_input_folder)

In [ ]:
# Stage 1: ZIP extraction with ETL logging
validation_message = "ZIP processing has not started"

# Expected file count from configuration
expected_files = int(expected_files)

try:
    # 1. Log execution start
    process_log("Start",file_name=file_name,layer="BRONZE")

    # 2. Validate and extract ZIP
    validation_message = unzip_files(final_input_folder,unzip_folder,expected_files)
    if "Error :" in validation_message:
        raise Exception(validation_message)
    print("Validation message after execution:", validation_message)

    # 3. Log successful execution
    process_log("Success",file_name=file_name,layer="BRONZE")

except Exception as e:
    # 4. Log failure
    process_log("Failed",file_name=file_name,layer="BRONZE",error_message=str(e))
    raise